# MAE DA Rollout — Reference File Opener

Utility notebook to open WoFS `.zarr.zip` reference files using the same pattern as `credit/data.py`'s `get_forward_data()`.

## 1. Import Libraries

In [ ]:
import time
from pathlib import Path

import xarray as xr

## 2. Define Simplified Zarr ZIP Opener

Mirrors the `get_forward_data()` logic from `credit/data.py`:
- Tries `consolidated=True` then `consolidated=False`
- Uses `zarr_format=2`, `decode_coords=False`, `create_default_indexes=False`
- Builds both `zip://<stem>::<path>` and `zip://<stem>/::<path>` URI variants
- Retries up to 3 times with back-off

In [ ]:
def open_wofs_zarr_zip(filepath, max_attempts: int = 3) -> xr.Dataset:
    """Open a WoFS .zarr.zip file produced by the CREDIT pipeline.

    Replicates the get_forward_data() logic in credit/data.py:
      - builds both zip URI variants (with and without trailing slash in root)
      - tries consolidated=True then consolidated=False
      - retries up to max_attempts times with exponential back-off

    Parameters
    ----------
    filepath : str or Path
        Path to the .zarr.zip file.
    max_attempts : int
        Number of retry attempts per URI.

    Returns
    -------
    xr.Dataset
    """
    zip_file = str(filepath).rstrip("/")
    zip_basename = Path(zip_file).stem  # e.g. "wofs_20200427_1700_mem01"

    # Two URI forms tried by data.py
    uris = [
        f"zip://{zip_basename}::{zip_file}",
        f"zip://{zip_basename}/::{zip_file}",
    ]

    def _try_open(uri: str) -> xr.Dataset:
        last_exc = None
        for consolidated in (True, False):
            try:
                return xr.open_zarr(
                    uri,
                    consolidated=consolidated,
                    zarr_format=2,
                    decode_coords=False,
                    create_default_indexes=False,
                )
            except Exception as exc:
                last_exc = exc
        raise last_exc

    def _try_with_retries(uri: str) -> xr.Dataset:
        last_exc = None
        for attempt in range(1, max_attempts + 1):
            try:
                return _try_open(uri)
            except Exception as exc:
                last_exc = exc
                if attempt < max_attempts:
                    time.sleep(0.25 * attempt)
        raise last_exc

    last_exc = None
    for uri in uris:
        try:
            return _try_with_retries(uri)
        except Exception as exc:
            last_exc = exc

    raise last_exc

## 3. Open and Inspect the Reference Dataset

In [ ]:
REF_FILE = Path(
    "/scratch5/purged/Zhanxiang.Hua/wofs_preprocess_to_credit_0413/cases/"
    "wofs_20200427_1700_mem01.zarr.zip"
)

ds_ref = open_wofs_zarr_zip(REF_FILE)
print(ds_ref)